# Importing Libraries 

In [1]:
import os
import pandas as pd 
import numpy as np 
from collections import defaultdict

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [2]:
df_games = pd.read_csv('../artifacts/interim_data/historical_matches.csv')

In [3]:
df_games.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [4]:
df_games['date'] = pd.to_datetime(df_games['date'])

In [5]:

df_games['tournament'].value_counts().head()

tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                            964
Name: count, dtype: int64

In [6]:
df_games['tournament'].value_counts().tail()

tournament
Copa Confraternidad               1
Benedikt Fontana Cup              1
ConIFA Challenger Cup             1
CONIFA World Cup qualification    1
South Asian Super Cup             1
Name: count, dtype: int64

In [7]:
df_tournaments = pd.DataFrame(df_games['tournament'].unique())

In [8]:
df_games['tournament'].str.startswith('CONIFA').sum()

np.int64(171)

# Feature Engineering

Let start creating the feature engineering for our prediction. We shall start with dome helper function. 

## Helper Functions 

We are dividing the matches into different tiers. Each tiers have different importance. Most important Tier is the world cup in which the best players play and Least importance are  Friendly matches where the best players might not be available. Some of helpers are related to World Cup for feature engineering. 

In [9]:
# K-factor tiers by tournament importance 
TIER_1_WORLD_CUP = {'FIFA World Cup'}

TIER_2_CONTINENTAL = {
    'UEFA Euro', 'Copa América', 'African Cup of Nations', 'AFC Asian Cup',
    'Gold Cup', 'CONCACAF Championship', 'Oceania Nations Cup',
    'Confederations Cup',
}

TIER_3_QUALIFIERS_NATIONS = {
    'FIFA World Cup qualification', 'UEFA Euro qualification',
    'African Cup of Nations qualification', 'AFC Asian Cup qualification',
    'Gold Cup qualification', 'CONCACAF Championship qualification',
    'Copa América qualification', 'Oceania Nations Cup qualification',
    'UEFA Nations League', 'CONCACAF Nations League',
    'CONCACAF Nations League qualification',
}

TIER_4_REGIONAL = {
    # Africa
    'CECAFA Cup', 'COSAFA Cup', 'COSAFA Cup qualification', 'WAFF Championship',
    'Amílcar Cabral Cup', 'All-African Games', 'UDEAC Cup', 'UNIFFAC Cup',
    'West African Cup', 'Nile Basin Tournament', 'African Friendship Games',
    # Asia / Oceania
    'Gulf Cup', 'Arab Cup', 'Arab Cup qualification', 'SAFF Cup',
    'AFF Championship', 'AFF Championship qualification', 'EAFF Championship',
    'EAFF Championship qualification', 'ASEAN Championship',
    'ASEAN Championship qualification', 'AFC Challenge Cup',
    'AFC Challenge Cup qualification', 'Asian Games', 'CAFA Nations Cup',
    'Southeast Asian Games', 'South Asian Games', 'Dynasty Cup',
    'Pacific Games', 'South Pacific Games', 'Melanesia Cup',
    'Indian Ocean Island Games', 'Afro-Asian Games',
    # Europe
    'British Home Championship', 'Nordic Championship', 'Baltic Cup',
    'Balkan Cup', 'Central European International Cup',
    # Americas
    'CFU Caribbean Cup', 'CFU Caribbean Cup qualification', 'UNCAF Cup',
    'Central American and Caribbean Games', 'Pan American Championship',
    'CCCF Championship', 'Bolivarian Games', 'NAFC Championship',
    # Multi-sport
    'Olympic Games',
}

TIER_5_FRIENDLY = {'Friendly', 'FIFA Series', 'CONCACAF Series'}

K_FACTOR_MAP = {
    **{t: 60 for t in TIER_1_WORLD_CUP},
    **{t: 50 for t in TIER_2_CONTINENTAL},
    **{t: 40 for t in TIER_3_QUALIFIERS_NATIONS},
    **{t: 30 for t in TIER_4_REGIONAL},
    **{t: 20 for t in TIER_5_FRIENDLY},
}
K_FACTOR_DEFAULT = 15
INITIAL_ELO      = 1500
HOME_ADVANTAGE   = 100

TOURNAMENT_STAGE_MAP = {
    'Group Stage':      1,
    'Round of 32':      2,
    'Round of 16':      3,
    'Quarter-finals':   4,
    'Semi-finals':      5,
    'Third-place match':6,
    'Final':            7,
}
 
CONFEDERATION_MAP = {
    # UEFA
    'Germany':'UEFA','France':'UEFA','Spain':'UEFA','Italy':'UEFA','England':'UEFA',
    'Portugal':'UEFA','Netherlands':'UEFA','Belgium':'UEFA','Croatia':'UEFA',
    'Switzerland':'UEFA','Denmark':'UEFA','Sweden':'UEFA','Poland':'UEFA',
    'Austria':'UEFA','Czechia':'UEFA','Serbia':'UEFA','Hungary':'UEFA',
    'Scotland':'UEFA','Ukraine':'UEFA','Turkey':'UEFA','Greece':'UEFA',
    'Romania':'UEFA','Slovakia':'UEFA','Slovenia':'UEFA','Norway':'UEFA',
    'Finland':'UEFA','Wales':'UEFA','Northern Ireland':'UEFA','Ireland':'UEFA',
    'Kosovo':'UEFA','Albania':'UEFA','Bosnia and Herzegovina':'UEFA',
    'North Macedonia':'UEFA','Montenegro':'UEFA','Iceland':'UEFA',
    'Georgia':'UEFA','Armenia':'UEFA','Azerbaijan':'UEFA','Moldova':'UEFA',
    'Belarus':'UEFA','Estonia':'UEFA','Latvia':'UEFA','Lithuania':'UEFA',
    'Luxembourg':'UEFA','Malta':'UEFA','Cyprus':'UEFA','Andorra':'UEFA',
    'San Marino':'UEFA','Liechtenstein':'UEFA','Gibraltar':'UEFA','Faroe Islands':'UEFA',
    # CONMEBOL
    'Brazil':'CONMEBOL','Argentina':'CONMEBOL','Uruguay':'CONMEBOL','Colombia':'CONMEBOL',
    'Chile':'CONMEBOL','Paraguay':'CONMEBOL','Ecuador':'CONMEBOL','Peru':'CONMEBOL',
    'Bolivia':'CONMEBOL','Venezuela':'CONMEBOL',
    # CONCACAF
    'United States':'CONCACAF','Mexico':'CONCACAF','Canada':'CONCACAF',
    'Costa Rica':'CONCACAF','Honduras':'CONCACAF','Jamaica':'CONCACAF',
    'Panama':'CONCACAF','El Salvador':'CONCACAF','Trinidad and Tobago':'CONCACAF',
    'Haiti':'CONCACAF','Cuba':'CONCACAF','Guatemala':'CONCACAF',
    # AFC
    'Japan':'AFC','South Korea':'AFC','Iran':'AFC','Saudi Arabia':'AFC',
    'Australia':'AFC','Qatar':'AFC','China':'AFC','Iraq':'AFC',
    'UAE':'AFC','Jordan':'AFC','Oman':'AFC','Bahrain':'AFC',
    'Uzbekistan':'AFC','Thailand':'AFC','Vietnam':'AFC','India':'AFC',
    # CAF
    'Morocco':'CAF','Senegal':'CAF','Nigeria':'CAF','Cameroon':'CAF',
    'Ghana':'CAF','Ivory Coast':'CAF','Tunisia':'CAF','Algeria':'CAF',
    'Egypt':'CAF','Mali':'CAF','South Africa':'CAF','Burkina Faso':'CAF',
    'DR Congo':'CAF','Guinea':'CAF','Zambia':'CAF','Zimbabwe':'CAF',
    # OFC
    'New Zealand':'OFC','Fiji':'OFC','Papua New Guinea':'OFC',
}
 
CONFEDERATION_STRENGTH = {
    'UEFA':     6,
    'CONMEBOL': 5,
    'CONCACAF': 3,
    'AFC':      3,
    'CAF':      3,
    'OFC':      1,
}
 
# World Cup host nations (year → host(s))
WC_HOSTS = {
    2026: ['United States', 'Canada', 'Mexico'],
    2022: ['Qatar'],
    2018: ['Russia'],
    2014: ['Brazil'],
    2010: ['South Africa'],
    2006: ['Germany'],
    2002: ['South Korea', 'Japan'],
    1998: ['France'],
    1994: ['United States'],
    1990: ['Italy'],
    1986: ['Mexico'],
    1982: ['Spain'],
    1978: ['Argentina'],
    1974: ['West Germany'],
    1970: ['Mexico'],
    1966: ['England'],
    1962: ['Chile'],
    1958: ['Sweden'],
    1954: ['Switzerland'],
    1950: ['Brazil'],
}
 
# World Cup winners & finalist history for knockout experience
WC_TITLES = {
    'Brazil': 5, 'Germany': 4, 'Italy': 4, 'Argentina': 3,
    'France': 2, 'Uruguay': 2, 'England': 1, 'Spain': 1,
}


DC_HALF_LIFE_DAYS = 180  
DC_MIN_MATCHES    = 5     
OUTPUT_DIR        = '../artifacts/processed_data'
FIFA_RANKINGS_PATH = '../artifacts/raw_data/wc_2026_48_teams_fifa_rank_change.csv'

 

## Pre Processing 

Let's do some preprocessing before moving the feature engineering section. We are removing the "CONIFA" tournament because the matches played in each continent is different and that will lead to get biased results. 

In [10]:


def preprocess_df(df: pd.DataFrame) -> pd.DataFrame:
    
    df = df.copy()         
    df = df.dropna(subset=['home_score', 'away_score'])
    df = df[~df['tournament'].str.contains('CONIFA', case=False, na=False)]
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)

    df['home_win']  = (df['home_score'] > df['away_score']).astype(int)
    df['away_win']  = (df['home_score'] < df['away_score']).astype(int)
    df['draw']      = (df['home_score'] == df['away_score']).astype(int)
    df['goal_diff'] = df['home_score'] - df['away_score']
    
    df['is_competitive'] = ~df['tournament'].isin(TIER_5_FRIENDLY)
    return df


## Elo ratings for every international team over history

We need define the Elo Ratings for the every international team over the history. We are keeping the default Elo rating as 1500  and based on home, away and neutral points are allocated. 

In [11]:

def get_k_factor(tournament):
    """K controls how reactive Elo is to a match. Higher = bigger swings."""
    return K_FACTOR_MAP.get(tournament, K_FACTOR_DEFAULT) # minor/exhibition tournaments, anniversary cups, etc.

def expected_score(home_elo, away_elo, home_advantage):
    # home_advantage = 100 → ~64% win rate for evenly matched home teams.
    # Set to 0 for neutral-venue matches (the formula then becomes symmetric).
    diff = (home_elo + home_advantage) - away_elo
    return 1 / (1 + 10 ** (-diff / 400))


def build_elo_history(df: pd.DataFrame):
    current_elo = defaultdict(lambda: INITIAL_ELO)
    elo_history = []

    for match in df.itertuples(index=False):
        home = match.home_team
        away = match.away_team

        home_elo = current_elo[home]
        away_elo = current_elo[away]

        h_adv = 0 if match.neutral else 100
        exp_home = expected_score(home_elo, away_elo, h_adv)
        exp_away = 1 - exp_home

        if match.home_score > match.away_score:
            actual_home, actual_away = 1.0, 0.0
        elif match.home_score < match.away_score:
            actual_home, actual_away = 0.0, 1.0
        else:
            actual_home, actual_away = 0.5, 0.5

        goal_diff = abs(match.home_score - match.away_score)

        if goal_diff <= 1:
            g = 1.0
        elif goal_diff == 2:
            g = 1.5
        else:
            g = (11 + goal_diff) / 8

        K = get_k_factor(match.tournament)

        current_elo[home] = home_elo + K * g * (actual_home - exp_home)
        current_elo[away] = away_elo + K * g * (actual_away - exp_away)

        elo_history.append((match.date, home, current_elo[home]))
        elo_history.append((match.date, away, current_elo[away]))

    df_elo = pd.DataFrame(
        elo_history,
        columns=["date", "team", "elo_after"]
    )

    df_elo["date"] = pd.to_datetime(df_elo["date"])

    return df_elo, current_elo    


##  ELO Lookup : Getting each team's Elo rating BEFORE a match

We are getting the recent Elo rating for team strictly before match_date. 

In [12]:
def get_elo_before_match(df_elo: pd.DataFrame, team: str, match_date: pd.Timestamp) -> float:
    """Return the most recent Elo rating for `team` strictly before `match_date`."""
    mask = (df_elo['team'] == team) & (df_elo['date'] < match_date)
    subset = df_elo[mask]
    if subset.empty:
        return 1500.0          # default for teams with no history
    return float(subset.iloc[-1]['elo_after'])
 

## Rolling Form:  last N matches per team

We are finding the Rolling Form for each team. 

In [13]:
def build_team_match_log(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Explode each row into two rows (one per team), capturing perspective-neutral stats.
    Returns a DataFrame sorted by (team, date).
    """
    home_rows = df_raw[[
        'date', 'home_team', 'away_team',
        'home_score', 'away_score', 'neutral', 'tournament'
    ]].rename(columns={
        'home_team': 'team', 'away_team': 'opponent',
        'home_score': 'goals_for', 'away_score': 'goals_against',
    })
    home_rows['is_home'] = 1
 
    away_rows = df_raw[[
        'date', 'away_team', 'home_team',
        'away_score', 'home_score', 'neutral', 'tournament'
    ]].rename(columns={
        'away_team': 'team', 'home_team': 'opponent',
        'away_score': 'goals_for', 'home_score': 'goals_against',
    })
    away_rows['is_home'] = 0
 
    log = pd.concat([home_rows, away_rows], ignore_index=True)
    log['win']  = (log['goals_for'] > log['goals_against']).astype(int)
    log['draw'] = (log['goals_for'] == log['goals_against']).astype(int)
    log['loss'] = (log['goals_for'] < log['goals_against']).astype(int)
    log['goal_diff'] = log['goals_for'] - log['goals_against']
    log['points'] = log['win'] * 3 + log['draw']
    log['clean_sheet'] = (log['goals_against'] == 0).astype(int)
    log['conceded_2plus'] = (log['goals_against'] >= 2).astype(int)
    log['scored_2plus']   = (log['goals_for']    >= 2).astype(int)
 
    return log.sort_values(['team', 'date']).reset_index(drop=True)
 
 

 
 
def rolling_form(team_log: pd.DataFrame,team: str, before_date: pd.Timestamp, n: int) -> dict:
    """Compute rolling form features for `team` over last `n` matches before `before_date`."""
    subset = team_log[
        (team_log['team'] == team) &
        (team_log['date'] < before_date)
    ].tail(n)
 
    prefix = f'last{n}_'
    if subset.empty:
        return {
            f'{prefix}matches':         0,
            f'{prefix}wins':            0,
            f'{prefix}draws':           0,
            f'{prefix}losses':          0,
            f'{prefix}goals_scored':    0.0,
            f'{prefix}goals_conceded':  0.0,
            f'{prefix}goal_diff':       0.0,
            f'{prefix}points':          0.0,
            f'{prefix}ppm':             0.0,      # points per match
            f'{prefix}clean_sheets':    0.0,
            f'{prefix}conceded_2plus':  0.0,
            f'{prefix}scored_2plus':    0.0,
            f'{prefix}win_rate':        0.0,
        }
 
    m = len(subset)
    return {
        f'{prefix}matches':        m,
        f'{prefix}wins':           subset['win'].sum(),
        f'{prefix}draws':          subset['draw'].sum(),
        f'{prefix}losses':         subset['loss'].sum(),
        f'{prefix}goals_scored':   subset['goals_for'].mean(),
        f'{prefix}goals_conceded': subset['goals_against'].mean(),
        f'{prefix}goal_diff':      subset['goal_diff'].sum(),
        f'{prefix}points':         subset['points'].sum(),
        f'{prefix}ppm':            subset['points'].mean(),
        f'{prefix}clean_sheets':   subset['clean_sheet'].mean(),
        f'{prefix}conceded_2plus': subset['conceded_2plus'].mean(),
        f'{prefix}scored_2plus':   subset['scored_2plus'].mean(),
        f'{prefix}win_rate':       subset['win'].mean(),
    }
 

##  DIXON-COLES Recency Weighting (exponential decay)

We are finding the recency weighting for the Dixon Coles model. 

In [14]:
 
def dc_attack_defense(team_log: pd.DataFrame,team: str, before_date: pd.Timestamp, half_life_days: float = 180.0, min_matches: int = 5) -> dict:
    """
    Compute exponentially-weighted attack / defense strength for Dixon-Coles.
 
    Weight formula:  w = exp(-λ * days_ago),  λ = ln(2) / half_life_days
    half_life_days=180  →  match 6 months ago has weight 0.5
    """
    lam = np.log(2) / half_life_days
 
    subset = team_log[
        (team_log['team'] == team) &
        (team_log['date'] < before_date)
    ].copy()
 
    if len(subset) < min_matches:
        return {
            'dc_attack_strength':  1.0,
            'dc_defense_strength': 1.0,
            'dc_weight_sum':       0.0,
        }
 
    subset['days_ago'] = (before_date - subset['date']).dt.days
    subset['weight']   = np.exp(-lam * subset['days_ago'])
 
    w_sum = subset['weight'].sum()
    wa_goals_for     = (subset['goals_for']     * subset['weight']).sum() / w_sum
    wa_goals_against = (subset['goals_against'] * subset['weight']).sum() / w_sum
 
    # Global weighted averages (used to normalise to a strength index)
    global_avg_scored   = team_log['goals_for'].mean()    or 1.0
    global_avg_conceded = team_log['goals_against'].mean() or 1.0
 
    return {
        'dc_attack_strength':  wa_goals_for     / global_avg_scored,
        'dc_defense_strength': wa_goals_against / global_avg_conceded,
        'dc_weight_sum':       w_sum,
    }
 

## Head to Head 

We are computing the head to head status of each team. 

In [15]:
def h2h_features(team_log: pd.DataFrame,team_a: str,team_b: str, before_date: pd.Timestamp, n: int = 10) -> dict:
    """Last `n` H2H encounters between team_a and team_b (from team_a's perspective)."""
 
    # Take the team_a perspective
    subset_a = team_log[
        (team_log['team'] == team_a) &
        (team_log['opponent'] == team_b) &
        (team_log['date'] < before_date)
    ].tail(n)
 
    empty = {
        'h2h_matches':       0,
        'h2h_wins':          0,
        'h2h_draws':         0,
        'h2h_losses':        0,
        'h2h_goals_scored':  0.0,
        'h2h_goals_conceded':0.0,
        'h2h_goal_diff':     0.0,
        'h2h_win_rate':      0.0,
        'h2h_last_result':   0,   # 1=win, 0=draw, -1=loss
    }
 
    if subset_a.empty:
        return empty
 
    m = len(subset_a)
    last_row = subset_a.iloc[-1]
 
    if last_row['win']:
        last_result = 1
    elif last_row['draw']:
        last_result = 0
    else:
        last_result = -1
 
    return {
        'h2h_matches':        m,
        'h2h_wins':           subset_a['win'].sum(),
        'h2h_draws':          subset_a['draw'].sum(),
        'h2h_losses':         subset_a['loss'].sum(),
        'h2h_goals_scored':   subset_a['goals_for'].mean(),
        'h2h_goals_conceded': subset_a['goals_against'].mean(),
        'h2h_goal_diff':      subset_a['goal_diff'].sum(),
        'h2h_win_rate':       subset_a['win'].mean(),
        'h2h_last_result':    last_result,
    }
 

# Tournament Intelligence 

We are finding Tournament based details. 

In [16]:
def tournament_features(team: str, match_date: pd.Timestamp, df_games: pd.DataFrame,
                         stage: str | None = None, host_nations: list[str] | None = None) -> dict:
    year = match_date.year

    hosts_this_wc = host_nations or WC_HOSTS.get(year, [])
    is_host = int(team in hosts_this_wc)

    confederation = CONFEDERATION_MAP.get(team, 'OTHER')
    conf_strength  = CONFEDERATION_STRENGTH.get(confederation, 2)

    stage_num = TOURNAMENT_STAGE_MAP.get(stage, 0) if stage else 0

    wc_matches = df_games[
        (df_games['tournament'] == 'FIFA World Cup') &
        ((df_games['home_team'] == team) | (df_games['away_team'] == team)) &
        (df_games['date'] < match_date)
    ]

    wc_appearances = wc_matches['date'].dt.year.nunique()

    if 'stage' in wc_matches.columns:
        wc_qf = wc_matches[wc_matches['stage'].isin(
            ['Quarter-finals', 'Semi-finals', 'Third-place match', 'Final']
        )].shape[0]
        wc_sf = wc_matches[wc_matches['stage'].isin(
            ['Semi-finals', 'Third-place match', 'Final']
        )].shape[0]
    else:
        wc_qf = 0
        wc_sf = 0

    return {
        'is_host':          is_host,
        'confederation':    confederation,
        'conf_strength':    conf_strength,
        'tournament_stage': stage_num,
        'wc_appearances':   wc_appearances,
        'wc_titles':        WC_TITLES.get(team, 0),
        'wc_qf_matches':    wc_qf,
        'wc_sf_matches':    wc_sf,
    }

## Rest and Fatigue (useful for live tournament)

Rest and team fatigue is an important factor in any major tournament. So we are calculating those features. 

In [17]:
def rest_features(team_log: pd.DataFrame, team: str, match_date: pd.Timestamp) -> dict:
    """Days since last match + number of matches in last 30 days."""
    recent = team_log[
        (team_log['team'] == team) &
        (team_log['date'] < match_date)
    ]
 
    if recent.empty:
        return {
            'days_since_last_match':    999,
            'matches_last_30_days':     0,
            'consecutive_matches':      0,
        }
 
    last_date   = recent['date'].max()
    days_rest   = (match_date - last_date).days
    last_30     = recent[recent['date'] >= match_date - pd.Timedelta(days=30)]
 
    # "consecutive matches" = unbroken run ending just before this match
    recent_sorted = recent.sort_values('date')
    dates = recent_sorted['date'].tolist()
    consecutive = 0
    for d in reversed(dates):
        gap = (match_date - d).days
        if gap <= 7:
            consecutive += 1
        else:
            break
 
    return {
        'days_since_last_match': days_rest,
        'matches_last_30_days':  len(last_30),
        'consecutive_matches':   consecutive,
    }

## Split the Feature List for each models 

As per are trying 3 different model. We need to used the features accordingly. 

In [18]:
def safe_feature_list(df: pd.DataFrame, feature_list: list[str]) -> list[str]:
    return [f for f in feature_list if f in df.columns]

## Main Feature Builder 

Let's build the Main Features

In [19]:
def build_match_features(row, df_elo: pd.DataFrame, team_log: pd.DataFrame,
                          df_games: pd.DataFrame, fifa_rankings: dict | None = None,
                          stage: str | None = None, host_nations: list[str] | None = None,
                          include_rest: bool = False) -> dict:
    date       = pd.Timestamp(row.date)
    home       = row.home_team
    away       = row.away_team
    is_neutral = int(row.neutral) if hasattr(row, 'neutral') else 0

    # Elo
    home_elo = get_elo_before_match(df_elo, home, date)
    away_elo = get_elo_before_match(df_elo, away, date)
    elo_diff = home_elo - away_elo

    # FIFA Rankings
    if fifa_rankings:
        home_rank = fifa_rankings.get(home, 999)
        away_rank = fifa_rankings.get(away, 999)
        rank_diff = away_rank - home_rank
    else:
        home_rank, away_rank, rank_diff = None, None, None

    # Rolling Form
    home_last3  = rolling_form(team_log, home, date, 3)
    home_last5  = rolling_form(team_log, home, date, 5)
    home_last10 = rolling_form(team_log, home, date, 10)
    away_last3  = rolling_form(team_log, away, date, 3)
    away_last5  = rolling_form(team_log, away, date, 5)
    away_last10 = rolling_form(team_log, away, date, 10)

    # Dixon-Coles
    home_dc = dc_attack_defense(team_log, home, date)
    away_dc = dc_attack_defense(team_log, away, date)

    # H2H
    h2h_home = h2h_features(team_log, home, away, date, n=10)
    h2h_away = h2h_features(team_log, away, home, date, n=10)

    # Tournament — now passes df_games explicitly
    t_home = tournament_features(home, date, df_games, stage, host_nations)
    t_away = tournament_features(away, date, df_games, stage, host_nations)

    # Rest & Fatigue
    r_home = rest_features(team_log, home, date) if include_rest else {}
    r_away = rest_features(team_log, away, date) if include_rest else {}

    # Assemble
    feats: dict = {
        'date':      date,
        'home_team': home,
        'away_team': away,
        'neutral':   is_neutral,
        'home_elo':  home_elo,
        'away_elo':  away_elo,
        'elo_diff':  elo_diff,
        'home_rank': home_rank,
        'away_rank': away_rank,
        'rank_diff': rank_diff,
    }

    for k, v in home_last3.items():  feats[f'home_{k}'] = v
    for k, v in home_last5.items():  feats[f'home_{k}'] = v
    for k, v in home_last10.items(): feats[f'home_{k}'] = v
    for k, v in away_last3.items():  feats[f'away_{k}'] = v
    for k, v in away_last5.items():  feats[f'away_{k}'] = v
    for k, v in away_last10.items(): feats[f'away_{k}'] = v

    for n in [3, 5, 10]:
        feats[f'diff_last{n}_goals_scored']   = feats[f'home_last{n}_goals_scored']   - feats[f'away_last{n}_goals_scored']
        feats[f'diff_last{n}_goals_conceded'] = feats[f'home_last{n}_goals_conceded'] - feats[f'away_last{n}_goals_conceded']
        feats[f'diff_last{n}_win_rate']       = feats[f'home_last{n}_win_rate']       - feats[f'away_last{n}_win_rate']
        feats[f'diff_last{n}_goal_diff']      = feats[f'home_last{n}_goal_diff']      - feats[f'away_last{n}_goal_diff']
        feats[f'diff_last{n}_ppm']            = feats[f'home_last{n}_ppm']            - feats[f'away_last{n}_ppm']

    for k, v in home_dc.items(): feats[f'home_{k}'] = v
    for k, v in away_dc.items(): feats[f'away_{k}'] = v
    feats['dc_attack_diff']     = home_dc['dc_attack_strength']  - away_dc['dc_attack_strength']
    feats['dc_defense_diff']    = home_dc['dc_defense_strength'] - away_dc['dc_defense_strength']

    for k, v in h2h_home.items(): feats[f'home_{k}'] = v
    for k, v in h2h_away.items(): feats[f'away_{k}'] = v

    for k, v in t_home.items(): feats[f'home_{k}'] = v
    for k, v in t_away.items(): feats[f'away_{k}'] = v
    feats['host_advantage']     = t_home['is_host']      - t_away['is_host']
    feats['conf_strength_diff'] = t_home['conf_strength'] - t_away['conf_strength']

    if include_rest:
        for k, v in r_home.items(): feats[f'home_{k}'] = v
        for k, v in r_away.items(): feats[f'away_{k}'] = v

    return feats

## Model Specific Features 

In [20]:
# ── Phase 1: Poisson baseline ────────────────────────────────
POISSON_FEATURES = [
    'elo_diff',
    'rank_diff',
    'neutral',
    'home_last5_goals_scored',
    'home_last5_goals_conceded',
    'away_last5_goals_scored',
    'away_last5_goals_conceded',
    'diff_last5_goals_scored',
    'diff_last5_goals_conceded',
    'home_last5_win_rate',
    'away_last5_win_rate',
    'home_h2h_win_rate',
    'away_h2h_win_rate',
    'home_h2h_goal_diff',
    'home_is_host',
    'away_is_host',
    'home_tournament_stage',
]
 
# ── Dixon-Coles features ─────────────────────────────────────
DIXON_COLES_FEATURES = [
    'home_dc_attack_strength',
    'home_dc_defense_strength',
    'away_dc_attack_strength',
    'away_dc_defense_strength',
    'dc_attack_diff',
    'dc_defense_diff',
    'home_dc_weight_sum',
    'away_dc_weight_sum',
    'elo_diff',
    'neutral',
    'home_is_host',
    'home_tournament_stage',
]
 
# ── XGBoost full feature set ─────────────────────────────────
XGBOOST_FEATURES = [
    # Elo & rankings
    'home_elo', 'away_elo', 'elo_diff',
    'home_rank', 'away_rank', 'rank_diff',
 
    # Context
    'neutral', 'host_advantage',
 
    # Last 3
    'home_last3_wins', 'home_last3_draws', 'home_last3_losses',
    'home_last3_goals_scored', 'home_last3_goals_conceded',
    'away_last3_wins', 'away_last3_draws', 'away_last3_losses',
    'away_last3_goals_scored', 'away_last3_goals_conceded',
    'diff_last3_goals_scored', 'diff_last3_goals_conceded', 'diff_last3_win_rate',
 
    # Last 5
    'home_last5_wins', 'home_last5_draws', 'home_last5_losses',
    'home_last5_goals_scored', 'home_last5_goals_conceded',
    'home_last5_goal_diff', 'home_last5_ppm', 'home_last5_win_rate',
    'home_last5_clean_sheets', 'home_last5_conceded_2plus', 'home_last5_scored_2plus',
    'away_last5_wins', 'away_last5_draws', 'away_last5_losses',
    'away_last5_goals_scored', 'away_last5_goals_conceded',
    'away_last5_goal_diff', 'away_last5_ppm', 'away_last5_win_rate',
    'away_last5_clean_sheets', 'away_last5_conceded_2plus', 'away_last5_scored_2plus',
    'diff_last5_goals_scored', 'diff_last5_goals_conceded',
    'diff_last5_win_rate', 'diff_last5_goal_diff', 'diff_last5_ppm',
 
    # Last 10
    'home_last10_goals_scored', 'home_last10_goals_conceded',
    'home_last10_goal_diff', 'home_last10_win_rate',
    'away_last10_goals_scored', 'away_last10_goals_conceded',
    'away_last10_goal_diff', 'away_last10_win_rate',
    'diff_last10_goals_scored', 'diff_last10_goals_conceded',
    'diff_last10_win_rate', 'diff_last10_goal_diff',
 
    # Dixon-Coles strength
    'home_dc_attack_strength', 'home_dc_defense_strength',
    'away_dc_attack_strength', 'away_dc_defense_strength',
    'dc_attack_diff', 'dc_defense_diff',
 
    # H2H
    'home_h2h_wins', 'home_h2h_draws', 'home_h2h_losses',
    'home_h2h_goals_scored', 'home_h2h_goals_conceded',
    'home_h2h_goal_diff', 'home_h2h_win_rate', 'home_h2h_last_result',
 
    # Tournament intelligence
    'home_is_host', 'away_is_host', 'host_advantage',
    'home_conf_strength', 'away_conf_strength', 'conf_strength_diff',
    'home_tournament_stage',
    'home_wc_appearances', 'away_wc_appearances',
    'home_wc_titles', 'away_wc_titles',
    'home_wc_qf_matches', 'away_wc_qf_matches',
]
 


## Building Full Dataset

Finally we are creating the full dataset and saving them as csv's.  

In [21]:
def build_feature_dataset(df_games: pd.DataFrame,fifa_rankings_path: str,output_dir: str = OUTPUT_DIR,include_rest: bool = False,host_nations: list[str] | None = None,) -> pd.DataFrame:
    """
    Full pipeline: preprocess → Elo → team log → features → save CSVs.
    Returns df_features (all features + targets in one DataFrame).
    """

    #  1. One-time setup (runs once, not per row) 
    print("Step 1/4: Preprocessing...")
    df = preprocess_df(df_games)

    print("Step 2/4: Building Elo history...")
    df_elo, _ = build_elo_history(df)

    print("Step 3/4: Building team match log...")
    team_log = build_team_match_log(df)

    #  2. FIFA Rankings
    fifa_rankings = (
        pd.read_csv(fifa_rankings_path)
        .set_index('Nation')['FIFA_2026_rank']
        .to_dict()
    )

    #  3. Build feature rows 
    print("\nStep 4/4: Building feature dataset...")
    records = []

    for row in df.itertuples(index=False):   
        feats = build_match_features(
            row,
            df_elo=df_elo,
            team_log=team_log,
            df_games=df_games,
            fifa_rankings=fifa_rankings,
            stage=getattr(row, 'stage', None),
            host_nations=host_nations,
            include_rest=include_rest,
        )
        # Targets
        feats['home_goals']     = row.home_score
        feats['away_goals']     = row.away_score
        feats['home_win']       = row.home_win        
        feats['away_win']       = row.away_win       
        feats['draw']           = row.draw           
        feats['goal_diff']      = row.goal_diff      
        feats['is_competitive'] = row.is_competitive 
        feats['result']         = (
            0 if row.home_score > row.away_score else
            1 if row.home_score == row.away_score else
            2
        )
        records.append(feats)

    df_features = pd.DataFrame(records)
    print(f"Feature dataset shape: {df_features.shape}")

    #  4. Model-specific slices → CSV 
    TARGET_COLS = ['date', 'home_team', 'away_team','home_goals', 'away_goals','home_win', 'away_win', 'draw', 'goal_diff', 'result','is_competitive',]

    poisson_cols  = safe_feature_list(df_features, POISSON_FEATURES)
    dc_cols       = safe_feature_list(df_features, DIXON_COLES_FEATURES)
    xgboost_cols  = safe_feature_list(df_features, XGBOOST_FEATURES)

    os.makedirs(output_dir, exist_ok=True)
    df_features[TARGET_COLS + poisson_cols ].to_csv(f'{output_dir}/poisson_features.csv',  index=False)
    df_features[TARGET_COLS + dc_cols      ].to_csv(f'{output_dir}/dc_features.csv',        index=False)
    df_features[TARGET_COLS + xgboost_cols ].to_csv(f'{output_dir}/xgboost_features.csv',  index=False)

    print(f"\nSaved to '{output_dir}':")
    print(f"  Poisson  : {len(poisson_cols)} features")
    print(f"  DC       : {len(dc_cols)} features")
    print(f"  XGBoost  : {len(xgboost_cols)} features")

    return df_features

# Main - Feature Engineering  

In [22]:

df_features = build_feature_dataset(
    df_games=df_games,
    fifa_rankings_path=FIFA_RANKINGS_PATH,
    output_dir=OUTPUT_DIR,
    include_rest=False,
)

Step 1/4: Preprocessing...
Step 2/4: Building Elo history...
Step 3/4: Building team match log...

Step 4/4: Building feature dataset...
Feature dataset shape: (49233, 155)

Saved to '../artifacts/processed_data':
  Poisson  : 17 features
  DC       : 12 features
  XGBoost  : 87 features


In [23]:
df_features.to_csv(f'{OUTPUT_DIR}/df_features.csv', index=False)